In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [2]:
df = pd.read_csv("UTS PEMSIN.csv", sep=";")

print("=== 1. INFORMASI DATASET AWAL ===")
print(f"Jumlah Baris dan Kolom : {df.shape}")
print(df[['REF_AREA_LABEL', 'TIME_PERIOD', 'OBS_VALUE']].head())

=== 1. INFORMASI DATASET AWAL ===
Jumlah Baris dan Kolom : (3756, 41)
  REF_AREA_LABEL  TIME_PERIOD            OBS_VALUE
0        Grenada         2019  107.888.374.763.244
1        Grenada         2023  102.789.696.954.297
2      Guatemala         2007  453.497.848.571.167
3      Guatemala         2010  234.500.978.194.729
4      Guatemala         2013  175.318.359.040.015


In [3]:
def clean_obs_value(val):
    if pd.isna(val):
        return np.nan
    v_str = str(val).strip()
    if v_str in ['0.0', '0']: # Menangani nilai murni nol
        return 0.0
    if 'E' in v_str: # Menangani outlier dengan scientific notation (misal 2,36E+08)
        return float(v_str.replace(',', '.'))
    
    # Menghapus titik ribuan yang corrupt (misal 107.888.374 -> 10.78)
    digits = v_str.replace('.', '').replace(',', '')
    if not digits.isdigit():
        return np.nan
    
    val_float = float(digits)
    # Menurunkan skala desimal menjadi rasio persentase yang wajar (Maks 100%)
    while val_float > 100:
        val_float /= 10
    return val_float

# Terapkan fungsi pembersihan
df['OBS_VALUE_CLEAN'] = df['OBS_VALUE'].apply(clean_obs_value)

In [4]:
df = df.dropna(subset=['OBS_VALUE_CLEAN'])
df_clean = df[df['OBS_VALUE_CLEAN'] <= 100].copy()

In [5]:
df_clean['Kategori_Tarif'] = np.where(df_clean['OBS_VALUE_CLEAN'] > 15.0, 'Tinggi', 'Rendah')

In [6]:
le = LabelEncoder()
df_clean['Kode_Negara_X2'] = le.fit_transform(df_clean['REF_AREA_LABEL'])

X = df_clean[['TIME_PERIOD', 'Kode_Negara_X2']]
y = df_clean['Kategori_Tarif']

print("\n=== 2. DATASET SETELAH PREPROCESSING ===")
print(df_clean[['TIME_PERIOD', 'Kode_Negara_X2', 'OBS_VALUE_CLEAN', 'Kategori_Tarif']].head())


=== 2. DATASET SETELAH PREPROCESSING ===
   TIME_PERIOD  Kode_Negara_X2  OBS_VALUE_CLEAN Kategori_Tarif
0         2019              68        10.788837         Rendah
1         2023              68        10.278970         Rendah
2         2007              69        45.349785         Tinggi
3         2010              69        23.450098         Tinggi
4         2013              69        17.531836         Tinggi


In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [9]:
knn = KNeighborsClassifier(n_neighbors=3, metric='euclidean')
knn.fit(X_train_scaled, y_train)

# Melakukan prediksi pada data uji
y_pred = knn.predict(X_test_scaled)